<a href="https://colab.research.google.com/github/seraphinemichelle/rm_project/blob/main/training_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaning sendiri

In [54]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

df = pd.read_csv('/content/drive/MyDrive/Dataset - RM.csv')

print("Shape data:", df.shape)
df.head()

# ===============================
# [1] Dataset Overview
# ===============================

print("[1] Dataset Overview")
print("-" * 60)

print(f"Total samples: {df.shape[0]}")
print(f"Features: {list(df.columns)}\n")

print("First 5 rows:")
print(df.head())

# ===============================
# [2] Class Distribution (Original)
# ===============================

print("\n[2] Class Distribution (Original)")
print("-" * 60)

class_counts = df['RiskLevel'].value_counts()
print(class_counts)

print("\nClass percentages:")
class_percent = df['RiskLevel'].value_counts(normalize=True) * 100

for label, percent in class_percent.items():
    print(f"{label}: {percent:.2f}%")

from sklearn.model_selection import train_test_split

# ==========================================
# [3] Data Preprocessing
# ==========================================

print("\n[3] Data Preprocessing")
print("-" * 60)

# Hapus NaN (kalau ada)
df_clean = df.dropna()

print(f"Samples after removing NaN: {df_clean.shape[0]}")

# Contoh konversi BodyTemp dari Fahrenheit ke Celcius (kalau memang perlu)
if df_clean['BodyTemp'].max() > 45:
  df_clean['BodyTemp'] = (df_clean['BodyTemp'] - 32) * 5/9

print(f"BodyTemp range after conversion: "
      f"{df_clean['BodyTemp'].min():.3f}C to {df_clean['BodyTemp'].max():.3f}C")

# Pisahkan fitur & target
X = df_clean.drop('RiskLevel', axis=1)
y = df_clean['RiskLevel']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

print("\nTraining set class distribution:")
print(y_train.value_counts())

from imblearn.over_sampling import SMOTE

print("\n[4] Applying SMOTE for Data Balancing")
print("-" * 60)

# Cek Distribusi Class Sebelum SMOTE
print("Training set class distribution before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nTraining set class distribution after SMOTE:")
print(y_train_smote.value_counts())

# ==========================================================
# [5] Hyperparameter Tuning with GridSearchCV
# ==========================================================

print("\n[5] Hyperparameter Tuning with GridSearchCV")
print("-" * 60)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(
    random_state=42,
    oob_score=True,
    bootstrap=True
)

# Grid yang stabil untuk dataset 2194
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [6, 8, 10],
    'max_features': ['sqrt'],
    'min_samples_split': [4, 6, 8],
    'min_samples_leaf': [2, 3]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_smote, y_train_smote)

print("\n[OK] Best parameters found:")
best_params = grid_search.best_params_

for param in best_params:
    print(f"{param}: {best_params[param]}")

print(f"\nBest CV F1-score (macro): {grid_search.best_score_:.4f}")

# ==========================================================
# [6] Training Final Model (Random Forest)
# ==========================================================

print("\n[6] Training Final Model")
print("-" * 60)

from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score

# Ambil model terbaik dari GridSearch
final_model = grid_search.best_estimator_

print("Best hyperparameters:")
for param, value in grid_search.best_params_.items():
    print(f"{param}: {value}")

# Train ulang pada data SMOTE
final_model.fit(X_train_smote, y_train_smote)

# ===============================
# OOB Score
# ===============================
if hasattr(final_model, "oob_score_"):
    print(f"\nOut-of-bag (OOB) Score: {final_model.oob_score_:.4f}")

# ===============================
# Cross Validation
# ===============================
cv_scores = cross_val_score(
    final_model,
    X_train_smote,
    y_train_smote,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

print(f"\nCross-validation accuracy: {cv_scores.mean():.4f} "
      f"(+/- {cv_scores.std():.4f})")

# ===============================
# Training vs Test Accuracy
# ===============================
train_pred = final_model.predict(X_train_smote)
test_pred = final_model.predict(X_test)

train_acc = accuracy_score(y_train_smote, train_pred)
test_acc = accuracy_score(y_test, test_pred)

gap = train_acc - test_acc

print("\nPerformance Check")
print("-" * 40)
print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Overfitting Gap: {gap:.4f}")

if gap < 0.10:
    print("[OK] Model generalizes well")
else:
    print("[WARNING] Potential overfitting detected")

print("\n[7] Model Evaluation")
print("-" * 60)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

# ===============================
# Overfitting Check
# ===============================

print("Overfitting Check")
print("-" * 40)

# Training accuracy
train_pred = final_model.predict(X_train_smote)
train_acc = accuracy_score(y_train_smote, train_pred)

# Test accuracy
test_pred = final_model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)

diff = train_acc - test_acc

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Difference: {diff:.4f}")

if diff < 0.10:
    print("[OK] Good generalization (low overfitting)")
else:
    print("[WARNING] Potential overfitting detected")

# ===============================
# Classification Report
# ===============================

print("\nClassification Report (Test Set)")
print("-" * 40)

print(classification_report(y_test, test_pred))

# ===============================
# Confusion Matrix (Formatted)
# ===============================

print("Confusion Matrix (Test Set)")
print("-" * 40)

labels = sorted(y.unique())
cm = confusion_matrix(y_test, test_pred, labels=labels)

cm_df = pd.DataFrame(cm, index=labels, columns=labels)

print(cm_df)

print("--- Per-Class Metrics ---")

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from sklearn.preprocessing import label_binarize

# Ambil metrik per kelas
precision, recall, f1, support = precision_recall_fscore_support(
    y_test,
    test_pred
)

labels = sorted(y.unique())

for i, label in enumerate(labels):
    print(f"\n{label}:")
    print(f"Precision: {precision[i]:.4f} "
          f"(of predicted {label}, {precision[i]*100:.2f}% were correct)")
    print(f"Recall:    {recall[i]:.4f} "
          f"(of actual {label}, {recall[i]*100:.2f}% were detected)")
    print(f"F1-score:  {f1[i]:.4f}")
    print(f"Support:   {support[i]} samples")

print("\n--- ROC AUC (macro, OVR) ---")

# Binarize label untuk multiclass ROC
y_test_bin = label_binarize(y_test, classes=labels)

# Probabilitas prediksi
y_prob = final_model.predict_proba(X_test)

roc_auc_macro = roc_auc_score(
    y_test_bin,
    y_prob,
    multi_class='ovr',
    average='macro'
)

print(f"Score: {roc_auc_macro:.4f}")

print("\n--- Feature Importance ---")

importances = final_model.feature_importances_
feature_names = X.columns

feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

for _, row in feat_imp.iterrows():
    print(f"{row['Feature']} : {row['Importance']:.4f} "
          f"({row['Importance']*100:.2f}%)")

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE")
print("="*60)

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

# Accuracy
y_pred_final = final_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_final)

# ROC AUC
labels = sorted(y.unique())
y_test_bin = label_binarize(y_test, classes=labels)
y_prob_final = final_model.predict_proba(X_test)

roc_auc = roc_auc_score(
    y_test_bin,
    y_prob_final,
    multi_class='ovr',
    average='macro'
)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"Total Trees (n_estimators): {final_model.n_estimators}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape data: (2194, 7)
[1] Dataset Overview
------------------------------------------------------------
Total samples: 2194
Features: ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel']

First 5 rows:
   Age  SystolicBP  DiastolicBP    BS  BodyTemp  HeartRate RiskLevel
0   25         130           80  15.0      98.0         86      High
1   35         140           90  13.0      98.0         70      High
2   29          90           70   8.0     100.0         80      High
3   30         140           85   7.0      98.0         70      High
4   35         120           60   6.1      98.0         76       Low

[2] Class Distribution (Original)
------------------------------------------------------------
RiskLevel
Low     1115
High     744
Mid      335
Name: count, dtype: int64

Class percentages:
Low: 50.82%
High: 33.91%
Mid: 15.27%

In [63]:
# ==========================================================
# [0] Load Dataset
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Dataset - RM.csv')

print("Dataset Shape:", df.shape)
print(df.head())

# ==========================================================
# [1] Dataset Overview
# ==========================================================

print("\n[1] Dataset Overview")
print("-" * 60)

print(f"Total samples: {df.shape[0]}")
print(f"Features: {list(df.columns)}")

print("\nClass Distribution:")
print(df['RiskLevel'].value_counts())

# ==========================================================
# [2] Data Cleaning & Encoding
# ==========================================================

print("\n[2] Data Preprocessing")
print("-" * 60)

# Remove missing values
df_clean = df.dropna().copy()

# Convert BodyTemp if in Fahrenheit
if df_clean['BodyTemp'].max() > 45:
    df_clean['BodyTemp'] = (df_clean['BodyTemp'] - 32) * 5/9

print("\nData Cleaning Summary")
print("-" * 60)
print(f"Jumlah data awal        : {df.shape[0]}")
print(f"Jumlah data setelah clean: {df_clean.shape[0]}")
print(f"Jumlah data terhapus    : {df.shape[0] - df_clean.shape[0]}")

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_clean['RiskLevel'] = label_encoder.fit_transform(df_clean['RiskLevel'])

print("\nClass Mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

X = df_clean.drop('RiskLevel', axis=1)
y = df_clean['RiskLevel']

# ==========================================================
# [3] Train-Test Split
# ==========================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

# ==========================================================
# [4] Model Setup
# ==========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(
        objective='multi:softprob',
        num_class=len(label_encoder.classes_),
        eval_metric='mlogloss',
        random_state=42
    )
}

# ==========================================================
# [5] Pipeline, Cross Validation, and Evaluation
# ==========================================================

from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

summary_results = []

labels = sorted(y.unique())
y_test_bin = label_binarize(y_test, classes=labels)

for name, model in models.items():

    print("\n" + "="*70)
    print(f"MODEL: {name}")
    print("="*70)

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    # Cross Validation
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy'
    )

    print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

    # Train full training set
    pipeline.fit(X_train, y_train)

    # Test prediction
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)

    test_acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')

    roc_auc = roc_auc_score(
        y_test_bin,
        y_prob,
        multi_class='ovr',
        average='macro'
    )

    print(f"Test Accuracy : {test_acc:.4f}")
    print(f"F1 Macro      : {f1_macro:.4f}")
    print(f"ROC AUC       : {roc_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    ))

    summary_results.append([
        name,
        test_acc,
        roc_auc,
        f1_macro,
        cv_scores.mean()
    ])

# ==========================================================
# [6] Final Model Comparison
# ==========================================================

summary_df = pd.DataFrame(
    summary_results,
    columns=[
        "Model",
        "Test Accuracy",
        "ROC AUC",
        "F1 Macro",
        "CV Accuracy"
    ]
)

summary_df = summary_df.sort_values(
    by="ROC AUC",
    ascending=False
).reset_index(drop=True)

print("\n" + "="*70)
print("FINAL MODEL COMPARISON")
print("="*70)
print(summary_df)

# ==========================================================
# [7] Best Model Interpretation
# ==========================================================

best_model = summary_df.iloc[0]

print("\n" + "="*70)
print("BEST MODEL INTERPRETATION")
print("="*70)

print(f"""
Model terbaik berdasarkan ROC AUC adalah {best_model['Model']}.

Test Accuracy : {best_model['Test Accuracy']:.4f}
ROC AUC       : {best_model['ROC AUC']:.4f}
F1 Macro      : {best_model['F1 Macro']:.4f}
CV Accuracy   : {best_model['CV Accuracy']:.4f}

Model ini menunjukkan performa terbaik dalam membedakan
seluruh kategori maternal risk secara seimbang.
""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Shape: (2194, 7)
   Age  SystolicBP  DiastolicBP    BS  BodyTemp  HeartRate RiskLevel
0   25         130           80  15.0      98.0         86      High
1   35         140           90  13.0      98.0         70      High
2   29          90           70   8.0     100.0         80      High
3   30         140           85   7.0      98.0         70      High
4   35         120           60   6.1      98.0         76       Low

[1] Dataset Overview
------------------------------------------------------------
Total samples: 2194
Features: ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel']

Class Distribution:
RiskLevel
Low     1115
High     744
Mid      335
Name: count, dtype: int64

[2] Data Preprocessing
------------------------------------------------------------

Data Cleaning Summary
----------------------------------

# Cleaning Code

In [64]:
# ==========================================================
# [0] Load Dataset
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Dataset.csv')

print("Dataset Shape:", df.shape)
print(df.head())

# ==========================================================
# [1] Dataset Overview
# ==========================================================

print("\n[1] Dataset Overview")
print("-" * 60)

print(f"Total samples: {df.shape[0]}")
print(f"Features: {list(df.columns)}")

print("\nClass Distribution:")
print(df['RiskLevel'].value_counts())

# ==========================================================
# [2] Data Cleaning & Encoding
# ==========================================================

print("\n[2] Data Preprocessing")
print("-" * 60)

# Remove missing values
df_clean = df.dropna().copy()

# Convert BodyTemp if in Fahrenheit
if df_clean['BodyTemp'].max() > 45:
    df_clean['BodyTemp'] = (df_clean['BodyTemp'] - 32) * 5/9

print("\nData Cleaning Summary")
print("-" * 60)
print(f"Jumlah data awal        : {df.shape[0]}")
print(f"Jumlah data setelah clean: {df_clean.shape[0]}")
print(f"Jumlah data terhapus    : {df.shape[0] - df_clean.shape[0]}")

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df_clean['RiskLevel'] = label_encoder.fit_transform(df_clean['RiskLevel'])

print("\nClass Mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

X = df_clean.drop('RiskLevel', axis=1)
y = df_clean['RiskLevel']

# ==========================================================
# [3] Train-Test Split
# ==========================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

# ==========================================================
# [4] Model Setup
# ==========================================================

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(
        objective='multi:softprob',
        num_class=len(label_encoder.classes_),
        eval_metric='mlogloss',
        random_state=42
    )
}

# ==========================================================
# [5] Pipeline, Cross Validation, and Evaluation
# ==========================================================

from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)
from sklearn.preprocessing import label_binarize

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

summary_results = []

labels = sorted(y.unique())
y_test_bin = label_binarize(y_test, classes=labels)

for name, model in models.items():

    print("\n" + "="*70)
    print(f"MODEL: {name}")
    print("="*70)

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    # Cross Validation
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy'
    )

    print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

    # Train full training set
    pipeline.fit(X_train, y_train)

    # Test prediction
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)

    test_acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')

    roc_auc = roc_auc_score(
        y_test_bin,
        y_prob,
        multi_class='ovr',
        average='macro'
    )

    print(f"Test Accuracy : {test_acc:.4f}")
    print(f"F1 Macro      : {f1_macro:.4f}")
    print(f"ROC AUC       : {roc_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    ))

    summary_results.append([
        name,
        test_acc,
        roc_auc,
        f1_macro,
        cv_scores.mean()
    ])

# ==========================================================
# [6] Final Model Comparison
# ==========================================================

summary_df = pd.DataFrame(
    summary_results,
    columns=[
        "Model",
        "Test Accuracy",
        "ROC AUC",
        "F1 Macro",
        "CV Accuracy"
    ]
)

summary_df = summary_df.sort_values(
    by="ROC AUC",
    ascending=False
).reset_index(drop=True)

print("\n" + "="*70)
print("FINAL MODEL COMPARISON")
print("="*70)
print(summary_df)

# ==========================================================
# [7] Best Model Interpretation
# ==========================================================

best_model = summary_df.iloc[0]

print("\n" + "="*70)
print("BEST MODEL INTERPRETATION")
print("="*70)

print(f"""
Model terbaik berdasarkan ROC AUC adalah {best_model['Model']}.

Test Accuracy : {best_model['Test Accuracy']:.4f}
ROC AUC       : {best_model['ROC AUC']:.4f}
F1 Macro      : {best_model['F1 Macro']:.4f}
CV Accuracy   : {best_model['CV Accuracy']:.4f}

Model ini menunjukkan performa terbaik dalam membedakan
seluruh kategori maternal risk secara seimbang.
""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Shape: (2219, 7)
   Age  SystolicBP  DiastolicBP    BS  BodyTemp  HeartRate RiskLevel
0   25       130.0         80.0  15.0      98.0       86.0      High
1   35       140.0         90.0  13.0      98.0       70.0      High
2   29        90.0         70.0   8.0     100.0       80.0      High
3   30       140.0         85.0   7.0      98.0       70.0      High
4   35       120.0         60.0   6.1      98.0       76.0       Low

[1] Dataset Overview
------------------------------------------------------------
Total samples: 2219
Features: ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel']

Class Distribution:
RiskLevel
Low     1119
High     746
Mid      336
Name: count, dtype: int64

[2] Data Preprocessing
------------------------------------------------------------

Data Cleaning Summary
----------------------------------